# Exploratory Analysis — GSE164416 T2D Pancreatic Islets

This notebook provides an interactive overview of the dataset and key results.
Run `bash run_pipeline.sh` before executing this notebook.

**Requirements:** All pipeline outputs must exist in `results/` and `data/processed/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score

plt.rcParams.update({'font.family': 'Arial', 'font.size': 11})
print('Ready.')

## 1. Cohort Overview

In [ ]:
labels = pd.read_csv('../data/processed/GSE164416_labels.csv')
print('Label distribution:')
print(labels['label'].value_counts().rename({1:'T2D', 0:'Non-diabetic', -1:'Excluded'}).to_string())
print(f'\nTotal samples: {len(labels)}')
print(f'Analyzed (T2D + ND): {(labels["label"].isin([0,1])).sum()}')

## 2. Gene Panel Summary

In [ ]:
panel = pd.read_csv('../results/final_gene_panel.csv')
deg   = pd.read_csv('../results/deg_results.csv')

sym_col = 'gene_symbol' if 'gene_symbol' in panel.columns else 'gene'
merged  = panel.merge(deg[['gene','log2FC','adj_pvalue','mean_T2D','mean_Ctrl']], on='gene', how='left')

display_cols = [sym_col, 'votes', 'composite_score', 'log2FC', 'adj_pvalue']
print(merged[[c for c in display_cols if c in merged.columns]].to_string(index=False))

## 3. Individual Gene AUC

In [ ]:
expr   = pd.read_csv('../data/processed/GSE164416_expr_normalized.csv', index_col=0)
ldf    = pd.read_csv('../data/processed/GSE164416_labels.csv')
ldf    = ldf[ldf['label'].isin([0,1])]
common = [c for c in expr.columns if c in ldf['sample_id'].values]
ldict  = dict(zip(ldf['sample_id'], ldf['label']))
y      = np.array([ldict[c] for c in common])

sym_map = dict(zip(panel['gene'], panel.get('gene_symbol', panel['gene'])))

aucs = []
for g in panel['gene'].tolist():
    if g not in expr.index: continue
    vals = expr.loc[g, common].values.astype(float)
    auc  = roc_auc_score(y, vals)
    auc  = max(auc, 1 - auc)
    aucs.append({'symbol': sym_map.get(g, g), 'auc': auc})

auc_df = pd.DataFrame(aucs).sort_values('auc', ascending=False)
print('Single-gene AUC values:')
for _, r in auc_df.iterrows():
    bar = '█' * int(r['auc'] * 30)
    print(f"  {r['symbol']:<15} {r['auc']:.4f}  {bar}")

## 4. LOOCV Performance

In [ ]:
try:
    loocv = pd.read_csv('../results/loocv_performance.csv').iloc[0]
    print('LOOCV Performance (Primary Validation)')
    print('='*45)
    for col in ['n_samples','n_t2d','n_nd','auc','sensitivity','specificity','f1','mcc','accuracy','brier']:
        if col in loocv.index:
            print(f'  {col:<20} {loocv[col]}')
except FileNotFoundError:
    print('Run: python 05_validation/loocv_validation.py')

## 5. Load and Display Pre-generated Figures

In [ ]:
from IPython.display import Image, display
import os

figures = [
    ('../figures/volcano_GSE164416.png',     'Fig 2: Volcano Plot'),
    ('../figures/venn_feature_overlap.png',  'Fig 3: Venn Diagram'),
    ('../figures/loocv_roc_curve.png',       'Fig 6: LOOCV ROC Curve'),
    ('../figures/individual_gene_auc.png',   'Fig 7: Individual Gene AUC'),
    ('../figures/expression_heatmap.png',    'Fig 5: Expression Heatmap'),
]

for path, title in figures:
    if os.path.exists(path):
        print(f'\n{title}')
        display(Image(path, width=700))
    else:
        print(f'  {title}: not found — run pipeline first')